In [20]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when, desc
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("Pertemuan7-StudiKasusTerintegrasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [22]:
spark = SparkSession.builder.appName("Latihan7").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi = spark.read.csv("hdfs://localhost:9000/user/maulzz/pertemuan7/raw/data_transaksi_triwulan.csv", header=True, inferSchema=True)
df_toko = spark.read.json("hdfs://localhost:9000/user/maulzz/pertemuan7/raw/data_toko.json")
df_produk = spark.read.csv("hdfs://localhost:9000/user/maulzz/pertemuan7/raw/data_produk.csv", header=True, inferSchema=True)
df_gabungan = df_transaksi.join(df_toko, on="toko_id").join(df_produk, on="product_id")
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))
print("Siap untuk latihan. Total baris:", df_gabungan.count())

Siap untuk latihan. Total baris: 5000


In [23]:
df_gabungan.createOrReplaceTempView("tabel_gabungan")

kueri_soal1 = """
SELECT 
    kategori, 
    SUM(qty) AS total_qty
FROM tabel_gabungan
GROUP BY kategori
ORDER BY total_qty DESC
"""

print("--- Soal 1: Kategori Produk dengan Total Qty Tertinggi ---")
spark.sql(kueri_soal1).show()

--- Soal 1: Kategori Produk dengan Total Qty Tertinggi ---
+----------+---------+
|  kategori|total_qty|
+----------+---------+
|Elektronik|     6736|
|   Fashion|     5307|
|   Makanan|     3114|
+----------+---------+



In [24]:
df_agregasi_toko = df_gabungan.groupBy("kategori", "nama_toko") \
    .agg(spark_sum("total_penjualan").alias("total_penjualan_toko"))

window_kategori = Window.partitionBy("kategori").orderBy(desc("total_penjualan_toko"))

df_top1_toko = df_agregasi_toko.withColumn("urut", row_number().over(window_kategori)) \
    .filter(col("urut") == 1) \
    .select("kategori", "nama_toko", "total_penjualan_toko")

print("--- Soal 2: Toko Penjualan Tertinggi per Kategori ---")
df_top1_toko.show()

--- Soal 2: Toko Penjualan Tertinggi per Kategori ---
+----------+-------------+--------------------+
|  kategori|    nama_toko|total_penjualan_toko|
+----------+-------------+--------------------+
|Elektronik|Toko Magelang|           202150000|
|   Fashion|Toko Semarang|           123000000|
|   Makanan|Toko Magelang|           108200000|
+----------+-------------+--------------------+



In [19]:
ringkasan_eksekutif = df_gabungan.groupBy("nama_toko", "kota", "manager").agg(
    spark_sum("total_penjualan").alias("total_penjualan_triwulan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
)

path_simpan = "hdfs://localhost:9000/user/maulzz/latihan7/ringkasan_eksekutif"

ringkasan_eksekutif.write \
    .mode("overwrite") \
    .parquet(path_simpan)

print(f"--- Soal 3: Data berhasil disimpan ke {path_simpan} ---")

--- Soal 3: Data berhasil disimpan ke hdfs://localhost:9000/user/maulzz/latihan7/ringkasan_eksekutif ---
